# Week 1 Practical: Exploring Molecules with RDKit
**AI for Drug Discovery**

In this practical, you will:
1. Install and import RDKit
2. Load a molecular dataset (EGFR inhibitors from ChEMBL)
3. Parse SMILES and visualize molecules
4. Calculate molecular properties
5. Plot property distributions
6. Apply Lipinski's Rule of Five

## 1. Setup and Installation
Run the cell below to install RDKit in Google Colab. If you're running locally, use `pip install rdkit`.

In [ ]:
# Install RDKit (Google Colab)
!pip install rdkit-pypi pandas matplotlib seaborn -q

import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Core imports
from rdkit import Chem
from rdkit.Chem import Draw, Descriptors, AllChem, Lipinski
from rdkit.Chem import PandasTools
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')
print('RDKit version:', Chem.rdBase.rdkitVersion)
print('Setup complete!')

## 2. Your First Molecule
Let's start by parsing a SMILES string and visualizing a molecule.

In [ ]:
# Parse a SMILES string into a molecule object
aspirin_smiles = 'CC(=O)Oc1ccccc1C(=O)O'
aspirin = Chem.MolFromSmiles(aspirin_smiles)

# Display the molecule
print(f'SMILES: {aspirin_smiles}')
print(f'Molecular Formula: {Chem.rdMolDescriptors.CalcMolFormula(aspirin)}')
print(f'Molecular Weight: {Descriptors.MolWt(aspirin):.2f} Da')

# Draw the molecule
Draw.MolToImage(aspirin, size=(400, 300))

In [ ]:
# Let's visualize several well-known drugs
drug_dict = {
    'Aspirin': 'CC(=O)Oc1ccccc1C(=O)O',
    'Caffeine': 'Cn1c(=O)c2c(ncn2C)n(C)c1=O',
    'Ibuprofen': 'CC(C)Cc1ccc(cc1)C(C)C(=O)O',
    'Penicillin V': 'CC1(C)S[C@@H]2[C@H](NC(=O)COc3ccccc3)C(=O)N2[C@@H]1C(=O)O',
    'Paracetamol': 'CC(=O)Nc1ccc(O)cc1',
    'Diazepam': 'CN1C(=O)CN=C(c2ccccc2)c2cc(Cl)ccc21'
}

mols = [Chem.MolFromSmiles(smi) for smi in drug_dict.values()]
legends = list(drug_dict.keys())

img = Draw.MolsToGridImage(mols, molsPerRow=3, subImgSize=(400, 300), legends=legends)
img

## 3. Load a Real Dataset: EGFR Inhibitors
We'll use a curated dataset of EGFR (Epidermal Growth Factor Receptor) inhibitors. EGFR is a major cancer drug target — drugs like gefitinib and erlotinib target EGFR.

We'll download data from ChEMBL or use a built-in dataset.

In [ ]:
# Load the Delaney solubility dataset (a classic benchmark)
# This is built into several cheminformatics packages
# For simplicity, we'll create a small dataset of known drugs and their properties

# Download ESOL (Delaney) dataset from DeepChem
try:
    url = 'https://raw.githubusercontent.com/deepchem/deepchem/master/datasets/delaney-processed.csv'
    df = pd.read_csv(url)
    print(f'Loaded {len(df)} molecules from Delaney solubility dataset')
    print(f'Columns: {list(df.columns)}')
    df.head()
except Exception as e:
    print(f'Could not download dataset: {e}')
    # Fallback: create a small dataset
    data = {
        'smiles': ['CCO', 'c1ccccc1', 'CC(=O)O', 'CCCCCCCC', 'OCC(O)CO',
                   'c1ccc2ccccc2c1', 'CC(C)O', 'CCCCCCCCCCC', 'OCc1ccccc1', 'CC(=O)Oc1ccccc1C(=O)O'],
        'measured log solubility in mols per litre': [-0.77, -0.77, 1.15, -3.26, 0.55,
                                                      -2.04, 0.25, -4.23, -0.54, -1.63]
    }
    df = pd.DataFrame(data)
    print(f'Using fallback dataset with {len(df)} molecules')

In [ ]:
# Parse all SMILES and add RDKit molecule objects
df['mol'] = df['smiles'].apply(lambda s: Chem.MolFromSmiles(s))
valid = df['mol'].notna()
print(f'Valid molecules: {valid.sum()} / {len(df)}')
df = df[valid].reset_index(drop=True)

# Show a grid of the first 12 molecules
mols_sample = df['mol'].head(12).tolist()
Draw.MolsToGridImage(mols_sample, molsPerRow=4, subImgSize=(300, 250))

## 4. Calculate Molecular Properties
RDKit can compute hundreds of molecular descriptors. Let's calculate the most important ones.

In [ ]:
# Calculate key molecular properties
df['MW'] = df['mol'].apply(Descriptors.MolWt)
df['LogP'] = df['mol'].apply(Descriptors.MolLogP)
df['HBD'] = df['mol'].apply(Descriptors.NumHDonors)
df['HBA'] = df['mol'].apply(Descriptors.NumHAcceptors)
df['TPSA'] = df['mol'].apply(Descriptors.TPSA)
df['RotBonds'] = df['mol'].apply(Descriptors.NumRotatableBonds)
df['AromaticRings'] = df['mol'].apply(Descriptors.NumAromaticRings)
df['HeavyAtoms'] = df['mol'].apply(Descriptors.HeavyAtomCount)

print('Molecular properties calculated!')
df[['smiles', 'MW', 'LogP', 'HBD', 'HBA', 'TPSA', 'RotBonds']].describe().round(2)

## 5. Visualize Property Distributions

In [ ]:
# Plot distributions of key properties
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
fig.suptitle('Distribution of Molecular Properties', fontsize=16, fontweight='bold')

props = ['MW', 'LogP', 'HBD', 'HBA', 'TPSA', 'RotBonds']
titles = ['Molecular Weight (Da)', 'LogP (Lipophilicity)', 'H-Bond Donors',
          'H-Bond Acceptors', 'Topological PSA', 'Rotatable Bonds']

for ax, prop, title in zip(axes.flat, props, titles):
    ax.hist(df[prop], bins=30, color='#1f77b4', edgecolor='white', alpha=0.7)
    ax.set_xlabel(title)
    ax.set_ylabel('Count')
    ax.axvline(df[prop].median(), color='red', linestyle='--', label=f'Median: {df[prop].median():.1f}')
    ax.legend()

plt.tight_layout()
plt.show()

## 6. Lipinski's Rule of Five
Let's check which molecules in our dataset pass Lipinski's Rule of Five (drug-likeness filter).

- MW <= 500
- LogP <= 5
- HBD <= 5
- HBA <= 10

In [ ]:
# Apply Lipinski's Rule of Five
def lipinski_pass(row):
    return (row['MW'] <= 500 and row['LogP'] <= 5 and
            row['HBD'] <= 5 and row['HBA'] <= 10)

df['Lipinski_Pass'] = df.apply(lipinski_pass, axis=1)

n_pass = df['Lipinski_Pass'].sum()
n_total = len(df)
print(f'Lipinski Rule of Five results:')
print(f'  Pass: {n_pass} ({100*n_pass/n_total:.1f}%)')
print(f'  Fail: {n_total - n_pass} ({100*(n_total-n_pass)/n_total:.1f}%)')

In [ ]:
# Visualize Lipinski passes vs fails
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
fig.suptitle("Lipinski Rule of Five Analysis", fontsize=14, fontweight='bold')

rules = [('MW', 500, 'MW <= 500'), ('LogP', 5, 'LogP <= 5'),
         ('HBD', 5, 'HBD <= 5'), ('HBA', 10, 'HBA <= 10')]

for ax, (prop, threshold, label) in zip(axes, rules):
    ax.hist(df[prop], bins=30, color='#1f77b4', edgecolor='white', alpha=0.7)
    ax.axvline(threshold, color='red', linestyle='--', linewidth=2, label=f'Threshold: {threshold}')
    ax.set_xlabel(label)
    ax.set_ylabel('Count')
    ax.legend()

plt.tight_layout()
plt.show()

## 7. Molecular Similarity
Let's compute Tanimoto similarity between molecules using Morgan fingerprints.

In [ ]:
from rdkit.Chem import AllChem
from rdkit import DataStructs

# Generate Morgan fingerprints for first 20 molecules
sample = df.head(20)
fps = [AllChem.GetMorganFingerprintAsBitVect(m, radius=2, nBits=2048) for m in sample['mol']]

# Compute pairwise Tanimoto similarity matrix
n = len(fps)
sim_matrix = np.zeros((n, n))
for i in range(n):
    for j in range(n):
        sim_matrix[i][j] = DataStructs.TanimotoSimilarity(fps[i], fps[j])

# Plot heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(sim_matrix, cmap='YlOrRd', vmin=0, vmax=1,
            xticklabels=range(n), yticklabels=range(n))
plt.title('Tanimoto Similarity Matrix (Morgan FP, radius=2)', fontsize=14)
plt.xlabel('Molecule Index')
plt.ylabel('Molecule Index')
plt.tight_layout()
plt.show()

print(f'Average pairwise similarity: {sim_matrix[np.triu_indices(n, k=1)].mean():.3f}')

## 8. Exercises

1. **Try different molecules**: Pick 3 drugs you know, find their SMILES on PubChem, and visualize them with RDKit
2. **Property comparison**: Calculate properties for your chosen drugs. Do they pass Lipinski's Rule of Five?
3. **Similarity search**: Which two molecules in the dataset are most similar? Which are most different?
4. **Challenge**: Can you find a molecule in the dataset that violates Lipinski's rules but is still a real drug?

## References
- Weininger, D. (1988). SMILES, a chemical language and information system. J. Chem. Inf. Comput. Sci. 28:31-36
- Lipinski, C.A. et al. (1997). Experimental and computational approaches to estimate solubility and permeability. Adv. Drug Delivery Rev. 23:3-25
- Gaulton, A. et al. (2017). The ChEMBL database in 2017. Nucleic Acids Research 45:D986-D994
- RDKit Documentation: https://www.rdkit.org/docs/